# 迭代

罗马不是一日建成的。同样，网络模型也不是一次训练就可以学会的。

在深度学习中，模型训练遵循一个朴素的规则：利用海量数据，通过成千上万次的**反复试错**，以**小步快跑**的方式逐渐逼近最佳模型参数。这种**重复且不断改进**的过程，称为**迭代**（Iteration）。

In [1]:
import numpy as np

## 张量

In [2]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data)
        self.grad = np.zeros_like(self.data)
        self.gradient_fn = None
        self.parents = set()

    def backward(self):
        if self.gradient_fn is not None:
            self.gradient_fn()

        for p in self.parents:
            p.backward()

    def __str__(self):
        return f'Tensor({self.data})'

## 数据集

模型训练所需要的海量数据，称为**数据集**（Dataset）。

数据集的每一条数据，称为一个**样本**（Sample）。每个样本都需要包括特征值和标签值两部分。

---

通常数据集会被分成两部分：
* **训练集**（Training Set）：较多的一部分（比如：80%），用于模型训练；
* **测试集**（Test Set）：另一部分较少的（比如：20%），用于模型评估。

划分数据集的目的是为了保留一部分**新数据**：测试集。这样，可以用**新数据**验证训练过的网络模型是否真正学会了规律，而不是只死记硬背了一些内容。这就像是期末考试，老师不会用你做过的练习题做考卷，而是用一套新题来考察你是不是真的掌握了学习的知识。

小明提供了过去四天的天气预报和他的销售记录，作为我们的训练集；我们之前使用的数据则作为测试集。

---

数据集通常会提供的函数包括：

* **load**（加载函数）：加载全部数据；
* **train**（训练函数）：切换到模型训练模式；
* **eval**（测试函数）：切换到模型测试模式；
* **all**（全部样本）：返回全部样本；
* **len**（样本数量）：返回样本数量；
* **getitem(index)**（单个样本）：根据索引（index）返回对应的样本。

In [3]:
class Dataset:

    def __init__(self):
        self.load()
        self.train()

    def load(self):
        self.train_data = ([[22.5, 72.0],
                            [31.4, 45.0],
                            [19.8, 85.0],
                            [27.6, 63.0]],
                           [[95],
                            [210],
                            [70],
                            [155]])
        self.test_data = ([[28.1, 58.0]],
                          [[165]])

    def train(self):
        self.data = self.train_data

    def eval(self):
        self.data = self.test_data

    def all(self):
        x, y = self.data
        return Tensor(x), Tensor(y)

    def __len__(self):
        x, *_ = self.data
        return len(x)

    def __getitem__(self, index):
        x, y = self.data
        return Tensor(x[index]), Tensor(y[index])

## 模型

In [4]:
class Linear:

    def __init__(self, in_size, out_size):
        self.weight = Tensor(np.ones((out_size, in_size)) / in_size)
        self.bias = Tensor(np.zeros(out_size))

    def __call__(self, x: Tensor):
        return self.forward(x)

    def forward(self, x: Tensor):
        p = Tensor(x.data @ self.weight.data.T + self.bias.data)

        def gradient_fn():
            self.weight.grad += p.grad * x.data
            self.bias.grad += np.sum(p.grad)

        p.gradient_fn = gradient_fn
        return p

    @property
    def parameters(self):
        return [self.weight, self.bias]

## 损失函数（均方误差）

In [5]:
class MSELoss:

    def __call__(self, p: Tensor, y: Tensor):
        return self.loss(p, y)

    def loss(self, p: Tensor, y: Tensor):
        mse = Tensor(np.mean(np.square(y.data - p.data)))

        def gradient_fn():
            p.grad += -2 * (y.data - p.data)

        mse.gradient_fn = gradient_fn
        mse.parents = {p}
        return mse

## 优化器（随机梯度下降）

为了适应模型训练时，将进行多次迭代的新需求，我们对优化器进行升级，增加了一个新的函数：
* **zero_grad**（清零函数）：每次迭代开始前，清零所有梯度，准备开始下一轮梯度计算。

In [6]:
class SGDOptimizer:

    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr

    def zero_grad(self):
        for p in self.parameters:
            p.grad = np.zeros_like(p.data)

    def step(self):
        for p in self.parameters:
            p.data -= p.grad * self.lr

## 超参数

### 学习率

In [7]:
LEARNING_RATE = 0.00001

## 建模

我们需要创建一个数据集的实例。

In [8]:
dataset = Dataset()
layer = Linear(2, 1)
loss_fn = MSELoss()
optimizer = SGDOptimizer(layer.parameters, lr=LEARNING_RATE)

## 训练

现在我们的数据集拥有多个训练样本，所以我们需要一个逻辑循环：每次从数据集中取出一个样本，完成一次迭代。

In [9]:
for i in range(len(dataset)):
    feature, label = dataset[i]

    optimizer.zero_grad()
    prediction = layer(feature)
    loss = loss_fn(prediction, label)
    loss.backward()
    optimizer.step()

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 2 is different from 1)

## 推理

模型训练完成以后，我们要将数据集转换成测试模式，用**新数据**（测试集）来评估模型训练的效果。

In [ ]:
dataset.eval()
feature, label = dataset.all()

prediction = layer(feature)
print(f'prediction:\t{prediction}')

## 评估

In [ ]:
loss = loss_fn(prediction, label)
print(f'loss:\t{loss}')

从验证结果来看，经过一轮迭代，损失值继续下降，网络模型的效果又有提高。但是仍然有很大的提升空间。

## 课后练习

训练数据迭代的次序会影响模型训练的效果吗？请尝试打乱训练数据的顺序（比如：改成逆序），再进行一轮迭代，观察结果有何不同。